<a href="https://colab.research.google.com/github/lilian662/SIMULACION-I/blob/main/SERPIENTES_Y_ESCALERAS%2C_RATON_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SERPIENTES Y ESCALERAS



Se modela el juego mediante una cadena de Markov y se construye un sistema lineal para calcular el número esperado de tiradas necesarias para llegar a la meta.

In [3]:
import numpy as np

# Número de casillas
N = 20

# Serpientes y escaleras
special = {
    3: 11,
    15: 19,
    13: 4,
    17: 9
}

# Función de transición

def move(pos, die):
    nxt = pos + die

    # Si se pasa de 20, se queda en 20
    if nxt > 20:
        nxt = 20

    # Aplicar serpiente o escalera
    if nxt in special:
        nxt = special[nxt]

    return nxt

# Construcción del sistema lineal
A = np.zeros((N, N))
b = np.ones(N)

for i in range(N):

    # Estado final
    if i == 19:
        A[i, i] = 1
        b[i] = 0
        continue

    A[i, i] = 1

    for die in range(1, 7):
        j = move(i + 1, die) - 1
        A[i, j] -= 1/6

# Resolver sistema
E = np.linalg.solve(A, b)

print("Esperanza de tiradas desde cada casilla:\n")

for i in range(N):
    print(f"Casilla {i+1}: {E[i]:.4f}")

print("\nNúmero promedio de tiradas iniciando en 1:")
print(E[0])

Esperanza de tiradas desde cada casilla:

Casilla 1: 7.0167
Casilla 2: 6.8068
Casilla 3: 6.8623
Casilla 4: 6.5261
Casilla 5: 6.2459
Casilla 6: 5.9364
Casilla 7: 6.0206
Casilla 8: 5.5472
Casilla 9: 4.8976
Casilla 10: 4.5090
Casilla 11: 4.5645
Casilla 12: 4.0791
Casilla 13: 3.1581
Casilla 14: 2.7069
Casilla 15: 2.5403
Casilla 16: 2.1774
Casilla 17: 1.3611
Casilla 18: 1.1667
Casilla 19: 1.0000
Casilla 20: 0.0000

Número promedio de tiradas iniciando en 1:
7.01671609272013


## Simulación Monte Carlo

Se simulan 100000 partidas para aproximar experimentalmente el número promedio de tiradas.

In [4]:
import numpy as np

special = {
    3: 11,
    15: 19,
    13: 4,
    17: 9
}


def simulate_game():

    pos = 1
    throws = 0

    while pos < 20:

        die = np.random.randint(1, 7)
        pos += die

        if pos > 20:
            pos = 20

        if pos in special:
            pos = special[pos]

        throws += 1

    return throws

# Número de simulaciones
M = 100000

results = [simulate_game() for _ in range(M)]

print("Promedio de tiradas:")
print(np.mean(results))

print("Desviación estándar:")
print(np.std(results))

Promedio de tiradas:
7.01018
Desviación estándar:
3.344651905295976


RATON



Se plantea un sistema de ecuaciones lineales basado en las probabilidades de transición del ratón entre casillas.

La solución del sistema permite obtener la probabilidad exacta de alcanzar la comida antes del shock eléctrico.

In [5]:
import numpy as np

# Variables: p0,p1,p2,p3,p4,p5,p6

A = np.array([
    [1, -1/2, -1/2, 0, 0, 0, 0],

    [-1/3, 1, 0, -1/3, 0, 0, 0],

    [-1/3, 0, 1, -1/3, 0, 0, 0],

    [0, -1/4, -1/4, 1, -1/4, -1/4, 0],

    [0, 0, 0, -1/3, 1, 0, -1/3],

    [0, 0, 0, -1/3, 0, 1, -1/3],

    [0, 0, 0, 0, -1/2, -1/2, 1]
])

b = np.array([
    0,
    1/3,
    0,
    0,
    1/3,
    0,
    0
])

p = np.linalg.solve(A,b)

print("Probabilidades:")

for i,val in enumerate(p):
    print(f"p{i} = {val:.6f}")

print("\nProbabilidad de alcanzar comida iniciando en 0:")
print(p[0])

Probabilidades:
p0 = 0.500000
p1 = 0.666667
p2 = 0.333333
p3 = 0.500000
p4 = 0.666667
p5 = 0.333333
p6 = 0.500000

Probabilidad de alcanzar comida iniciando en 0:
0.49999999999999994


## Simulación Monte Carlo

Para validar el resultado analítico, se realizan múltiples simulaciones del movimiento aleatorio del ratón.

En cada simulación el ratón se mueve aleatoriamente entre casillas vecinas hasta alcanzar la comida o recibir la descarga eléctrica.

In [6]:
import numpy as np

neighbors = {
    0: [1,2],
    1: [0,3,7],
    2: [0,3,8],
    3: [1,2,4,5],
    4: [3,6,7],
    5: [3,6,8],
    6: [4,5],
    7: [],
    8: []
}


def simulate_mouse():

    state = 0

    while True:

        if state == 7:
            return 1

        if state == 8:
            return 0

        state = np.random.choice(neighbors[state])

# Número de simulaciones
M = 100000

results = [simulate_mouse() for _ in range(M)]

print("Probabilidad estimada:")
print(np.mean(results))

Probabilidad estimada:
0.50216


En el problema de serpientes y escaleras, el juego fue modelado mediante una cadena de Markov, donde cada casilla representa un estado y las serpientes y escaleras modifican las transiciones entre estados. A partir del sistema de ecuaciones obtenido se calculó el número esperado de tiradas necesarias para llegar a la meta. Posteriormente, mediante simulación Monte Carlo, se reprodujo el juego miles de veces y se obtuvo un promedio muy cercano al resultado analítico, validando así el modelo matemático desarrollado.

Por otro lado, en el problema del ratón se estudió la probabilidad de alcanzar la comida antes de recibir una descarga eléctrica. Se construyó un sistema de ecuaciones basado en las probabilidades de transición entre estados del tablero, considerando a la comida y al shock como estados absorbentes. La solución analítica permitió obtener la probabilidad exacta de éxito iniciando desde la casilla 0, mientras que la simulación Monte Carlo confirmó experimentalmente dicho resultado.